# DateWise: AI-Powered Speed Dating and Strategic Matchmaking

## First-project modelling notebook

**Objective:** Explore whether information available before a speed date can help estimate whether a pair will mutually match.

This notebook documents the complete first modelling attempt, from raw-data inspection to a held-out evaluation of logistic regression. The emphasis is on a transparent and reproducible workflow—not on claiming that the model is already accurate enough for real-world matchmaking.

### Project scope
- Dataset: the Kaggle speed-dating dataset (`Speed Dating Data.csv`).
- Prediction target: `match` (0 = no mutual match, 1 = mutual match).
- Model family: logistic regression only.
- Preprocessing: median imputation and standardisation.
- Validation strategy: split by `wave` so that waves do not overlap between training and testing.

> **Before running:** Keep `Speed Dating Data.csv` in the same folder as this notebook, or update the path in the data-loading cell below.

## 1. Imports

All libraries used in this notebook are imported here so dependencies are easy to review and the code can be run from a fresh kernel.

In [ ]:
# Data handling
import numpy as np
import pandas as pd

# Data splitting and preprocessing
from sklearn.model_selection import GroupShuffleSplit
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

# Model
from sklearn.linear_model import LogisticRegression

# Evaluation metrics
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report,
    precision_score,
    recall_score,
    f1_score,
    average_precision_score,
    roc_auc_score,
)

## 2. Load and inspect the dataset

We load the Kaggle speed-dating dataset and inspect its dimensions, sample records, summary statistics, data types, and column names. The file is read with `cp1252` encoding to handle its character encoding correctly.

In [ ]:
df = pd.read_csv(
    "Speed Dating Data.csv",
    encoding="cp1252"
    )

print("Dataset dimensions:")
print(df.shape)

print("\nFirst five rows:")
print(df.head())

print("\nRandom five rows:")
print(df.sample(5, random_state=42))

print("\nStatistical summary:")
print(df.describe(include='all').T)

print("\nDataset information:")
df.info()

print("\nDataset columns:")
print(df.columns.to_list())

## 3. Initial data-quality checks

We check for exact duplicate rows and review the candidate variables that may be available before a date. The feature report records data types, missingness, and the number of distinct values.

The exploratory preference checks below examine whether the six stated partner-preference scores approximately add up to 100. These checks help us understand the data; they do not automatically remove observations.

In [ ]:
print(df.duplicated().sum())

In [ ]:
# Revised Time 1 candidate features

candidate_features = [
    # Demographic information
    "gender", "age", "age_o",
    "field_cd", "career_c", "goal", "date", "go_out",

    # Lifestyle interests
    "sports", "tvsports", "exercise", "dining",
    "museums", "art", "hiking", "gaming",
    "clubbing", "reading", "tv", "theater",
    "movies", "concerts", "music", "shopping", "yoga",

    # Expectations before the event
    "exphappy",

    # Self-assessment
    "attr3_1", "sinc3_1", "intel3_1",
    "fun3_1", "amb3_1",

    # Personal preferences
    "attr1_1", "sinc1_1", "intel1_1",
    "fun1_1", "amb1_1", "shar1_1",

    # Partner's stated preferences
    "pf_o_att", "pf_o_sin", "pf_o_int",
    "pf_o_fun", "pf_o_amb", "pf_o_sha",

    # Pre-date interest correlation
    "int_corr"
]

# Feature report
feature_report = pd.DataFrame({
    "dtype": df[candidate_features].dtypes,
    "missing_count": df[candidate_features].isnull().sum(),
    "missing_percentage": (
        df[candidate_features].isnull().mean() * 100
    ).round(2),
    "unique_values": df[candidate_features].nunique()
})

print("FEATURE REPORT")
print(feature_report.sort_values(
    "missing_percentage",
    ascending=False
).to_string())


print("\nWAVE COUNTS:")
print(df["wave"].value_counts().sort_index())

### 3.1 Inspecting stated partner preferences

The six `attr1_1`–`shar1_1` columns describe the importance participants assign to partner attributes. We calculate their row-wise total only when all six values are present, then inspect the totals and individual columns by wave.

In [ ]:
# Six attributes participants value in a partner
preference_cols = [
    "attr1_1",
    "sinc1_1",
    "intel1_1",
    "fun1_1",
    "amb1_1",
    "shar1_1"
]

# Sum the six preference values.
# min_count=6 ensures incomplete rows are excluded.
df["preference_sum"] = df[preference_cols].sum(
    axis=1,
    min_count=6
)
# Examine preference totals separately for each wave
wave_preference_report = df.groupby("wave")[
    "preference_sum"
].agg([
    "count",
    "mean",
    "std",
    "min",
    "median",
    "max"
])

print(wave_preference_report.to_string())

In [ ]:
preference_cols = [
    "attr1_1", "sinc1_1", "intel1_1",
    "fun1_1", "amb1_1", "shar1_1"
]

# Examine individual preference values in each wave
wave_attribute_report = df.groupby("wave")[preference_cols].agg(
    ["count", "mean", "min", "max"]
)

print(wave_attribute_report.to_string())

In [ ]:
# Check preference totals outside the expected range
unusual = df[
    df["preference_sum"].notna() &
    ((df["preference_sum"] < 99) |
     (df["preference_sum"] > 101))
]

print("Number of unusual records:", len(unusual))

print("\nUnusual records by wave:")
print(unusual.groupby("wave").size())

print("\nPreference total distribution:")
print(df["preference_sum"].describe())

## 4. Constructing reciprocal pair records

The raw data is organised as directed interactions: a row for participant A evaluating participant B is distinct from the reverse row for B evaluating A.

For a mutual-match task, we want one record per unordered pair. We first inspect whether reverse interactions exist and whether the two directions agree on the `match` label. We then retain one direction per reciprocal pair (`iid < pid`) to avoid counting the same pair twice.

In [ ]:
pair_check = df.merge(
    df[["iid", "pid"]],
    left_on=["pid", "iid"],
    right_on=["iid", "pid"],
    how="left",
    indicator=True
)

print(pair_check["_merge"].value_counts())

In [ ]:
# Keep only records that have a reciprocal interaction
reciprocal = df.merge(
    df[["iid", "pid", "match"]],
    left_on=["pid", "iid"],
    right_on=["iid", "pid"],
    how="inner",
    suffixes=("", "_reverse")
)

# Check whether match labels agree
inconsistent_matches = reciprocal[
    reciprocal["match"] != reciprocal["match_reverse"]
]

print("Reciprocal records:", len(reciprocal))
print("Inconsistent match labels:", len(inconsistent_matches))

In [ ]:
# Select and rename columns for checking reverse interactions
reverse_records = df[["iid", "pid"]].rename(
    columns={
        "iid": "iid_reverse",
        "pid": "pid_reverse"
    }
)

# Keep only records that have a reverse interaction
reciprocal_df = df.merge(
    reverse_records,
    left_on=["pid", "iid"],
    right_on=["iid_reverse", "pid_reverse"],
    how="inner"
)

# Keep only one direction of each pair
pairs_df = reciprocal_df[
    reciprocal_df["iid"] < reciprocal_df["pid"]
].copy()

print("Original directed records:", len(df))
print("Reciprocal directed records:", len(reciprocal_df))
print("Unique pair-level records:", len(pairs_df))
print(
    "Unique participant pairs:",
    pairs_df[["iid", "pid"]].drop_duplicates().shape[0]
)

## 5. Building participant-level and pair-level features

We select participant attributes that are intended to be known before the date. One row is retained per participant, and we verify that the selected participant-level values are consistent across that participant's records.

Next, each pair is represented symmetrically:
- **Numerical variables:** pair mean and absolute difference.
- **Categorical variables:** an indicator for whether both participants share the same category.

This representation avoids making the feature vector depend on which participant happened to be listed first. Missing categorical values are preserved as missing rather than being treated as a disagreement.

In [ ]:
# Select participant-level features
participant_features = [
    "gender",
    "age",
    "field_cd",
    "career_c",
    "goal",
    "date",
    "go_out",
    "sports",
    "tvsports",
    "exercise",
    "dining",
    "museums",
    "art",
    "hiking",
    "gaming",
    "clubbing",
    "reading",
    "tv",
    "theater",
    "movies",
    "concerts",
    "music",
    "shopping",
    "yoga",
    "exphappy",
    "attr1_1",
    "sinc1_1",
    "intel1_1",
    "fun1_1",
    "amb1_1",
    "shar1_1",
    "attr3_1",
    "sinc3_1",
    "intel3_1",
    "fun3_1",
    "amb3_1"
]

# Keep one record per participant
participants = (
    df[["iid"] + participant_features]
    .drop_duplicates(subset="iid")
    .copy()
)

print("Number of participants:", len(participants))
print("Unique participant IDs:", participants["iid"].nunique())

In [ ]:
consistency_check = (
    df.groupby("iid")[participant_features]
    .nunique()
)

inconsistent_columns = consistency_check.columns[
    (consistency_check > 1).any()
]

print("Inconsistent participant-level columns:")
print(inconsistent_columns.tolist())

In [ ]:
participants_A = participants.rename(
    columns={col: f"{col}_A" for col in participant_features}
)
participants_B = participants.rename(
    columns={
        "iid": "pid",
        **{col: f"{col}_B" for col in participant_features}
    }
)

pair_features = pairs_df[["iid", "pid", "match"]].copy()
pair_features = pair_features.merge(
    participants_A, 
    on="iid",
    how = "left",
    validate = "many_to_one"
)

pair_features = pair_features.merge(
    participants_B,
    on="pid",
    how="left",
    validate="many_to_one"
)

print("Pair-level dataset shape:", pair_features.shape)
print("\nFirst five rows:")
print(pair_features.head())

In [ ]:
print(
    "Unique pairs:",
    pair_features[["iid", "pid"]].drop_duplicates().shape[0]
)

print(
    "Missing A features:\n",
    pair_features[
        [f"{col}_A" for col in participant_features]
    ].isnull().sum().sum()
)

print(
     "Missing B features:",
    pair_features[
        [f"{col}_B" for col in participant_features]
    ].isnull().sum().sum()
)

print(pair_features["match"].value_counts())
print(pair_features["match"].value_counts(normalize=True) * 100)

In [ ]:
# Keep pair identifiers and target
pair_model_df = pair_features[["iid", "pid", "match"]].copy()

# Categorical features
categorical_features = [
    "gender",
    "field_cd",
    "career_c",
    "goal"
]

# All remaining features are treated as numerical
numeric_features = [
    col for col in participant_features
    if col not in categorical_features
]

# Create symmetric numerical features
for col in numeric_features:

    A = pair_features[f"{col}_A"]
    B = pair_features[f"{col}_B"]

    pair_model_df[f"{col}_mean"] = (A + B) / 2

    pair_model_df[f"{col}_diff"] = (A - B).abs()


# Create symmetric categorical features
for col in categorical_features:

    A = pair_features[f"{col}_A"]
    B = pair_features[f"{col}_B"]

    same = (A == B).astype(float)

    # Preserve missing values rather than treating them as differences
    same[A.isna() | B.isna()] = float("nan")

    pair_model_df[f"{col}_same"] = same


print("Original pair dataset:", pair_features.shape)
print("Symmetric feature dataset:", pair_model_df.shape)

print(pair_model_df.head())

In [ ]:
# Check missing values in our engineered features
missing = pair_model_df.isnull().sum()

print("Features with missing values:")
print(missing[missing > 0].sort_values(ascending=False))


# Check features with only one unique value
print("\nUnique values in each feature:")

for col in pair_model_df.columns:
    if col not in ["iid", "pid", "match"]:
        print(col, ":", pair_model_df[col].nunique(dropna=False))

In [ ]:
pair_model_df = pair_model_df.drop(columns=["gender_same"])

### 5.1 Wave-level exploration

We check how many waves participants appear in and summarise the number of pairs and match rate in each wave. This matters because observations from the same event wave may share context. The modelling split therefore keeps entire waves together.

In [ ]:
# Count how many different waves each participant appears in
wave_check = df.groupby("iid")["wave"].nunique()

# Count participants belonging to 1, 2, 3, etc. waves
print(wave_check.value_counts().sort_index())

In [ ]:
print(
    pairs_df.groupby("wave")
    .size()
    .sort_index()
)

In [ ]:
wave_stats = (
    pairs_df.groupby("wave")["match"]
    .agg(["count", "sum", "mean"])
)

wave_stats = wave_stats.rename(columns={
    "count": "total_pairs",
    "sum": "successful_matches",
    "mean": "match_rate"
})

wave_stats["match_rate"] *= 100

print(wave_stats.round(2))

## 6. Train/test split by wave

We attach the wave identifier to each pair and use `GroupShuffleSplit` with `wave` as the grouping variable.

This is stricter than a random row split: no wave can appear in both training and testing. It gives us a test of whether the model transfers to held-out waves, rather than merely recognising patterns from waves it has already seen.

In [ ]:
pair_model_df = pair_model_df.merge(
    pairs_df[["iid", "pid", "wave"]],
    on=["iid", "pid"],
    how="left",
    validate="one_to_one"
)

In [ ]:
# Separate features and target
X = pair_model_df.drop(columns=["iid", "pid", "match", "wave"])
y = pair_model_df["match"]

# Use wave as the grouping variable
groups = pair_model_df["wave"]

# Create group-based train/test split
splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(X, y, groups=groups)
)

# Create training and testing datasets
X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]

y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

In [ ]:
print("Training features:", X_train.shape)
print("Testing features:", X_test.shape)

print("\nTraining target:", y_train.shape)
print("Testing target:", y_test.shape)

print("\nTraining match distribution:")
print(y_train.value_counts())

print("\nTesting match distribution:")
print(y_test.value_counts())

In [ ]:
train_waves = set(groups.iloc[train_idx])
test_waves = set(groups.iloc[test_idx])

print("Training waves:", sorted(train_waves))
print("Testing waves:", sorted(test_waves))

print("Common waves:", train_waves.intersection(test_waves))

## 7. Preprocessing and baseline logistic regression

The engineered features are numerical. Missing values are filled using the **training-set median**, and features are standardised. The preprocessing pipeline is fitted on training data only, then applied to the test data.

We first fit a standard logistic regression model with the default class weighting. Because matches are the minority class, accuracy alone can be misleading; we inspect the confusion matrix and per-class precision, recall, and F1-score.

In [ ]:
# Build preprocessing steps for numerical pair features.
# The imputer and scaler are fitted on training data only.
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

In [ ]:
# Fit preprocessing steps using training data only.
# Apply the fitted transformations to both datasets.
X_train_processed = numeric_pipeline.fit_transform(X_train)
X_test_processed = numeric_pipeline.transform(X_test)

print("Training shape:", X_train_processed.shape)
print("Testing shape:", X_test_processed.shape)
print("Missing values in training:", np.isnan(X_train_processed).sum())
print("Missing values in testing:", np.isnan(X_test_processed).sum())

In [ ]:
model = LogisticRegression(max_iter=1000)
model.fit(X_train_processed, y_train)

In [ ]:
y_pred = model.predict(X_test_processed)
print("First 20 predictions:", y_pred[:20])
print("Total predictions:", len(y_pred))

In [ ]:
accuracy = accuracy_score(y_test, y_pred)

print("Accuracy:", accuracy)
print("Accuracy (%):", accuracy * 100)

In [ ]:
cm = confusion_matrix(y_test, y_pred)

print(cm)

In [ ]:
print(classification_report(y_test, y_pred))

In [ ]:
y_prob = model.predict_proba(X_test_processed)[:, 1]

print("First 20 probabilities:", y_prob[:20])
print("\nProbability statistics:")
print(pd.Series(y_prob).describe())

## 8. Validation-based threshold and class-weight experiments

The positive class (`match = 1`) is relatively uncommon. A default probability threshold of 0.5 may therefore miss many actual matches.

To explore this, we create a second wave-grouped split **inside the training data**. We compare:
1. The original logistic regression model at several probability thresholds.
2. Logistic regression with `class_weight="balanced"` at several thresholds.

The validation set is used for these choices. The held-out test set remains separate for the final evaluation.

In [ ]:
groups_train = groups.iloc[train_idx]
validation_splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=42
)

train_inner_idx, val_idx = next(
    validation_splitter.split(
        X_train,
        y_train,
        groups=groups_train
    )
)
X_train_inner = X_train.iloc[train_inner_idx]
X_val = X_train.iloc[val_idx]

y_train_inner = y_train.iloc[train_inner_idx]
y_val = y_train.iloc[val_idx]
print("Inner training:", X_train_inner.shape)
print("Validation:", X_val.shape)
print("Common waves:", set(groups_train.iloc[train_inner_idx]) &
      set(groups_train.iloc[val_idx]))

In [ ]:
validation_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])
X_train_inner_processed = validation_pipeline.fit_transform(
    X_train_inner
)
X_val_processed = validation_pipeline.transform(X_val)

In [ ]:
validation_model = LogisticRegression(max_iter=1000)

validation_model.fit(
    X_train_inner_processed,
    y_train_inner
)
y_val_prob = validation_model.predict_proba(
    X_val_processed
)[:, 1]
print(pd.Series(y_val_prob).describe())

In [ ]:
thresholds = [0.5, 0.4, 0.3, 0.2, 0.1]

for threshold in thresholds:

    y_val_pred = (y_val_prob >= threshold).astype(int)

    precision = precision_score(y_val, y_val_pred, zero_division=0)
    recall = recall_score(y_val, y_val_pred, zero_division=0)
    f1 = f1_score(y_val, y_val_pred, zero_division=0)

    print(f"\nThreshold: {threshold}")
    print(f"Precision: {precision:.3f}")
    print(f"Recall:    {recall:.3f}")
    print(f"F1-score:  {f1:.3f}")

In [ ]:
for threshold in [0.2, 0.1]:

    y_val_pred = (y_val_prob >= threshold).astype(int)

    print(f"\nThreshold: {threshold}")
    print("Confusion matrix:")
    print(confusion_matrix(y_val, y_val_pred))

    print("Precision:", precision_score(y_val, y_val_pred))
    print("Recall:", recall_score(y_val, y_val_pred))
    print("F1:", f1_score(y_val, y_val_pred))

In [ ]:
balanced_model = LogisticRegression(
    max_iter=1000,
    class_weight="balanced"
)
balanced_model.fit(
    X_train_inner_processed,
    y_train_inner
)
y_val_prob_balanced = balanced_model.predict_proba(
    X_val_processed
)[:, 1]
print(pd.Series(y_val_prob_balanced).describe())

In [ ]:
thresholds = [0.5, 0.4, 0.3, 0.2, 0.1]

for threshold in thresholds:

    y_val_pred_balanced = (
        y_val_prob_balanced >= threshold
    ).astype(int)

    precision = precision_score(
        y_val, y_val_pred_balanced, zero_division=0
    )

    recall = recall_score(
        y_val, y_val_pred_balanced, zero_division=0
    )

    f1 = f1_score(
        y_val, y_val_pred_balanced, zero_division=0
    )

    print(f"\nThreshold: {threshold}")
    print(f"Precision: {precision:.3f}")
    print(f"Recall:    {recall:.3f}")
    print(f"F1-score:  {f1:.3f}")

In [ ]:
y_val_pred_balanced = (
    y_val_prob_balanced >= 0.4
).astype(int)

cm_balanced = confusion_matrix(
    y_val,
    y_val_pred_balanced
)

print(cm_balanced)

In [ ]:
ap_original = average_precision_score(
    y_val,
    y_val_prob
)

ap_balanced = average_precision_score(
    y_val,
    y_val_prob_balanced
)

print("Original model AP:", ap_original)
print("Balanced model AP:", ap_balanced)

## 9. Final held-out test evaluation

The validation experiments suggest using a balanced logistic regression model with a probability threshold of 0.4. We now train a fresh pipeline on the complete training split and evaluate it once on the held-out test waves.

The threshold is a decision rule applied to predicted probabilities:
- probability ≥ 0.4 → predict a mutual match;
- probability < 0.4 → predict no mutual match.

This threshold favours finding more positive cases, at the cost of more false positives.

In [ ]:
# Create fresh pipeline
final_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        max_iter=1000,
        class_weight="balanced"
    ))
])

# Train on all original training data
final_pipeline.fit(X_train, y_train)

# Get probabilities for the positive class
y_test_prob = final_pipeline.predict_proba(X_test)[:, 1]

# Apply the selected threshold
y_test_pred = (y_test_prob >= 0.4).astype(int)

# Evaluate
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_test_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_test_pred))

print("Average Precision:", average_precision_score(y_test, y_test_prob))
print("ROC-AUC:", roc_auc_score(y_test, y_test_prob))

## 10. Results and interpretation

### What the final test result tells us

The held-out test evaluation should be interpreted using the confusion matrix and the positive-class metrics, not accuracy alone.

- **Recall for matches** measures the fraction of actual mutual matches that the model identifies.
- **Precision for matches** measures the fraction of predicted matches that are truly mutual matches.
- **Average precision** summarises the precision–recall trade-off across thresholds.
- **ROC-AUC** measures ranking discrimination across thresholds.

A high recall with low precision means the model finds many actual matches but also recommends many pairs that do not mutually match. This is not yet a reliable matchmaking system.

### Limitations and next steps

1. **Limited model scope:** This first project uses logistic regression, as required by the project constraints. More expressive models have not been explored.
2. **Class imbalance:** Mutual matches are a minority class, so accuracy can look acceptable even when the model misses most matches.
3. **Feature limitations:** The engineered features summarise similarity and participant preferences, but may not capture interaction dynamics, context, or nonlinear compatibility.
4. **Generalisation:** A wave-based split is a useful stricter test, but performance can vary depending on which waves are held out.
5. **Threshold trade-off:** Lowering the threshold increases recall but also increases false positives. A matchmaking product would need to decide how costly each type of error is.
6. **Historical dataset:** Results on this dataset do not establish that the model will work for a modern dating platform or for users outside the study.

### Conclusion

This project establishes an end-to-end baseline for strategic matchmaking: raw interactions are converted into unique reciprocal pairs, participant attributes are transformed into symmetric pair features, and a logistic regression pipeline is trained and evaluated without mixing event waves across splits.

The current model is a **first baseline, not a deployment-ready recommender**. Its main value is demonstrating the complete machine-learning workflow and revealing the difficulty of predicting mutual decisions from limited pre-date information.